In [ ]:
# Lab6 참고 — 체크포인트/산출물 경로를 프로젝트 루트 기준으로 고정
from pathlib import Path
import sys

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "project_paths.py").is_file():
        sys.path.insert(0, str(_p))
        break
from project_paths import project_root, artifacts_lab6

ROOT = project_root()
ART_LAB6 = artifacts_lab6()

In [1]:
import re
import torch
from pathlib import Path
from dataclasses import dataclass, field

### FIXED-POINT SPECIFICATION: ap_fixed<16, 7>
FIXED_W = 16
FIXED_I = 7
FIXED_F = FIXED_W - FIXED_I

FIXED_SCALE = 2 ** FIXED_F
FIXED_MIN = -(2 ** (FIXED_I - 1))
FIXED_MAX = (2 ** (FIXED_I - 1)) - (1 / FIXED_SCALE)

def fixed_point_round_tensor(x):
    """
    Emulate signed ap_fixed<16, 7> for exporting weights.
    Range: [-64, 64)
    Step size: 2^-9
    """
    x = torch.clamp(x, FIXED_MIN, FIXED_MAX)
    x = torch.round(x * FIXED_SCALE) / FIXED_SCALE
    return x

@dataclass(init=True)
class Checkpoint:
    path: str
    keys: list[str] = field(init=False, default_factory=list)
    values: list[torch.Tensor] = field(init=False, default_factory=list)

    def __post_init__(self):
        loaded = torch.load(self.path, map_location='cpu')

        for key, value in loaded['model'].items():
            self.keys.append(key)
            self.values.append(value)

    @staticmethod
    def _make_good_name(name):
        name = re.sub(f"[^0-9a-zA-Z_]", '_', name)

        if name and name[0].isdigit():
            name = "_" + name
        return name.upper()

    @classmethod
    def _nested_format(cls, obj, fmt, indent=0):
        sp = "\t" * indent

        if isinstance(obj, list):
            if not obj:
                return "{}"
            if not isinstance(obj[0], list):
                return "{" + ", ".join(fmt(x) for x in obj) + "}"
            inner = tuple(cls._nested_format(x, fmt, indent+1) for x in obj)
            return "{\n" + ",\n".join("\t" * (indent+1) + s for s in inner) + "\n" + sp + "}"
        else:
            return fmt(obj)

    def export_fixed(self, output_path):
        type_name = "fixed_t"

        lines = [
            "#pragma once",
            "#include <cstdint>",
            "#include <cstddef>",
            "#include \"ap_fixed.h\"",
            "",
            f"using {type_name} = ap_fixed<{FIXED_W}, {FIXED_I}, AP_RND, AP_SAT>;",
            "",
        ]

        def fmt(x):
            return f"{repr(float(x))}f"

        for name, tsr in zip(self.keys, self.values):
            name = self._make_good_name(name)

            tsr = tsr.detach().cpu().float().contiguous()
            tsr = fixed_point_round_tensor(tsr)

            shape = tuple(tsr.shape)
            dims = "".join(f"[{d}]" for d in shape)
            nested = tsr.tolist()

            lines.extend([
                f"// {name} shape={shape}, dtype=ap_fixed<{FIXED_W}, {FIXED_I}>",
                *[f"#define {name}_DIM{i} {d}" for i, d in enumerate(shape)],
                f"static const {type_name} {name}{dims} = {self._nested_format(nested, fmt, 0)};",
                f""
            ])

        with open(output_path, 'w', encoding='utf-8') as h:
            h.write('\n'.join(lines))

In [3]:
checkpoint = Checkpoint(str(ROOT / 'milestones_qat' / 'MNISTSLP-2026-05-12-09-39-19.pth'))
checkpoint.export_fixed(str(ART_LAB6 / 'weights.h'))